In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

Cloning into 'CTAB-GAN-Plus'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 77 (delta 21), reused 17 (delta 17), pack-reused 48 (from 1)
Unpacking objects: 100% (77/77), 1.05 MiB | 7.15 MiB/s, done.


In [2]:
pip install sdv

Note: you may need to restart the kernel to use updated packages.


In [3]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers and sample data
data = data.replace("?", np.nan)

# Complete-case only — no mean / mode imputation
_before = len(data)
_n_missing_rows = int(data.isna().any(axis=1).sum())
data = data.dropna().reset_index(drop=True)
print(
    f"Dropped {_before - len(data)} rows with missing feature values "
    f"(complete-case; no imputation; rows_with_na={_n_missing_rows})"
)
assert not data.isna().any().any(), "Unexpected NaNs remain after dropna"

# Sample after complete-case so NA handling is not confounded by imputation
n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)
print(f"Adult subsample after complete-case: {len(data)} rows")

numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare features, target, and metadata
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

# Set experiment constants and seeds
N_SAMPLES = 10000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Initialize result containers
scores = {}
synthetic_datasets = {}
quality_results = []


Dropped 3620 rows with missing feature values (complete-case; no imputation; rows_with_na=3620)
Adult subsample after complete-case: 1000 rows


In [4]:
# Single run

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Train/test split without leakage

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
    stratify=processed_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# CTABGAN

try:
    data_path = "adult_train.csv"
    train_real.to_csv(data_path, index=False)

    categorical_columns = [
        col for col in train_real.columns
        if col != target_col and col in label_encoders
    ]

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=categorical_columns + [target_col],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Classification": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    # Ensure the target column in CTABGAN synthetic data is integer-encoded (0 or 1)
    # consistent with the real_data. The `label_encoders` dictionary from previous cells
    # contains the LabelEncoder fitted on the original string labels.
    if target_col in synthetic_ctabgan.columns:
        # Convert to numeric, coercing errors to NaN. This handles object dtype containing numeric strings.
        synthetic_ctabgan[target_col] = pd.to_numeric(synthetic_ctabgan[target_col], errors='coerce')

        # Fill any NaNs that might have been introduced by coercion
        # Filling with the mode of the real target column for consistency.
        real_target_mode = train_real[target_col].mode()[0]
        synthetic_ctabgan[target_col] = synthetic_ctabgan[target_col].fillna(real_target_mode)

        # Convert to integer type
        synthetic_ctabgan[target_col] = synthetic_ctabgan[target_col].astype(int)

        # Ensure values are strictly 0 or 1 by clipping them
        synthetic_ctabgan[target_col] = synthetic_ctabgan[target_col].clip(0, 1)

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)


================ SINGLE RUN ================


100%|██████████| 150/150 [00:36<00:00,  4.15it/s]


Finished training in 39.028074502944946  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 43.57it/s]|
Column Shapes Score: 33.75%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 212.77it/s]|
Column Pair Trends Score: 0.0%

Overall Score (Average): 16.87%

CTABGAN: 0.1687


In [5]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    encoder = LabelEncoder()
    data_wgan[target_col] = encoder.fit_transform(data_wgan[target_col])

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_wgan[target_col] = (
        synthetic_wgan[target_col]
        .round()
        .clip(0, 1)
        .astype(int)
    )

    # Removed the following line as it converts target column back to strings,
    # which causes issues with evaluate_quality when compared to integer-encoded real data.
    # synthetic_wgan[target_col] = encoder.inverse_transform(
    #     synthetic_wgan[target_col]
    # )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:08<00:00,  1.67it/s]|
Column Shapes Score: 45.02%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 231.95it/s]|
Column Pair Trends Score: 0.0%

Overall Score (Average): 22.51%

WGAN_GP: 0.2251


In [6]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 56.44it/s]|
Column Shapes Score: 85.16%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 324.78it/s]|
Column Pair Trends Score: 72.48%

Overall Score (Average): 78.82%

CTGAN: 0.7882
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 55.12it/s]|
Column Shapes Score: 80.89%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 282.49it/s]|
Column Pair Trends Score: 72.06%

Overall Score (Average): 76.48%

CopulaGAN: 0.7648
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 68.73it/s]|
Column Shapes Score: 80.4%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 292.76it/s]|
Column Pair Trends Score: 59.17%

Overall Score (Average): 69.79%

TVAE: 0.6979
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 37.20it/s]|
Column Shapes

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None
):
    if seeds is None:
        seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            # Extract X and y from the full training DataFrame for the current seed
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]

            # Extract X and y from the full testing DataFrame for the current seed
            X_test_full = test_df.drop(columns=[label_col])
            y_test_full = test_df[label_col]

            # Determine if stratification is possible for y_train_full
            stratify_y_train_full = y_train_full if y_train_full.value_counts().min() >= 2 else None
            if stratify_y_train_full is None:
                print(
                    f"Warning: Cannot stratify training data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `train_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the first train-test split (from train_df to get actual training set for classifier)
            X_train_split, _, y_train_split, _ = train_test_split(
                X_train_full,
                y_train_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_train_full
            )

            # Determine if stratification is possible for y_test_full
            # This is usually for real data and should be fine, but included for robustness.
            stratify_y_test_full = y_test_full if y_test_full.value_counts().min() >= 2 else None
            if stratify_y_test_full is None:
                print(
                    f"Warning: Cannot stratify testing data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `test_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the second train-test split (from test_df to get actual testing set for classifier)
            _, X_test_split, _, y_test_split = train_test_split(
                X_test_full,
                y_test_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_test_full
            )

            scaler = StandardScaler().fit(X_train_split)

            X_train_s = scaler.transform(X_train_split)
            X_test_s = scaler.transform(X_test_split)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train_split)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test_split, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

        acc_mean = np.mean(accuracy_scores)
        acc_std = np.std(accuracy_scores, ddof=1)

        f1_mean = np.mean(f1_scores)
        f1_std = np.std(f1_scores, ddof=1)

        prec_mean = np.mean(precision_scores)
        prec_std = np.std(precision_scores, ddof=1)

        rec_mean = np.mean(recall_scores)
        rec_std = np.std(recall_scores, ddof=1)

        results.append({
            "Model": name,

            "Accuracy Mean": acc_mean,
            "Accuracy Std": acc_std,
            "F1 Mean": f1_mean,
            "F1 Std": f1_std,
            "Precision Mean": prec_mean,
            "Precision Std": prec_std,
            "Recall Mean": rec_mean,
            "Recall Std": rec_std,

            "Accuracy \u00b1 SD": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 \u00b1 SD": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision \u00b1 SD": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall \u00b1 SD": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}",

            "Accuracy (Mean\u00b1Std)": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 (Mean\u00b1Std)": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision (Mean\u00b1Std)": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall (Mean\u00b1Std)": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )

In [9]:
# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers
data = data.replace("?", np.nan)

# Clean income target into two classes only
data[target_col] = (
    data[target_col]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.strip()
)

print(data[target_col].value_counts())

# Complete-case only — no mean / mode imputation
_before = len(data)
_n_missing_rows = int(data.isna().any(axis=1).sum())
data = data.dropna().reset_index(drop=True)
print(
    f"Dropped {_before - len(data)} rows with missing feature values "
    f"(complete-case; no imputation; rows_with_na={_n_missing_rows})"
)
assert not data.isna().any().any(), "Unexpected NaNs remain after dropna"

n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)
print(f"Adult subsample after complete-case: {len(data)} rows")

numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare processed dataset
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

print("\nEncoded income classes:")
print(processed_data[target_col].value_counts())
print(label_encoders[target_col].classes_)


income
<=50K    37155
>50K     11687
Name: count, dtype: int64
Dropped 3620 rows with missing feature values (complete-case; no imputation; rows_with_na=3620)
Adult subsample after complete-case: 1000 rows

Encoded income classes:
income
0    728
1    272
Name: count, dtype: int64
['<=50K' '>50K']


In [10]:
# TRTR evaluation for Adult income dataset

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

X = processed_data.drop(columns=[target_col])
y = processed_data[target_col]

print("Target column:", target_col)
print("Target classes:")
print(y.value_counts())

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores, ddof=1)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores, ddof=1)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores, ddof=1)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores, ddof=1)

    trtr_results.append({
        "Model": model_name,

        "Accuracy Mean_TRTR": acc_mean,
        "Accuracy Std_TRTR": acc_std,
        "F1 Mean_TRTR": f1_mean,
        "F1 Std_TRTR": f1_std,
        "Precision Mean_TRTR": prec_mean,
        "Precision Std_TRTR": prec_std,
        "Recall Mean_TRTR": rec_mean,
        "Recall Std_TRTR": rec_std,

        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results).sort_values(
    by="Accuracy Mean_TRTR",
    ascending=False
)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)

--- Starting TRTR Evaluation (Train Real, Test Real) ---
Target column: income
Target classes:
income
0    728
1    272
Name: count, dtype: int64
Running LogReg...
Running SVM-RBF...
Running KNN...
Running NaiveBayes...
Running DecisionTree...
Running RandomForest...
Running ExtraTrees...
Running GradientBoost...
Running AdaBoost...
Running MLP...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
7,GradientBoost,0.8490 ± 0.0248,0.8433 ± 0.0277,0.8445 ± 0.0273,0.8490 ± 0.0248
8,AdaBoost,0.8390 ± 0.0221,0.8313 ± 0.0261,0.8334 ± 0.0242,0.8390 ± 0.0221
5,RandomForest,0.8375 ± 0.0308,0.8322 ± 0.0344,0.8323 ± 0.0344,0.8375 ± 0.0308
6,ExtraTrees,0.8225 ± 0.0333,0.8156 ± 0.0363,0.8157 ± 0.0369,0.8225 ± 0.0333
3,NaiveBayes,0.7850 ± 0.0215,0.7516 ± 0.0273,0.7787 ± 0.0310,0.7850 ± 0.0215
4,DecisionTree,0.7815 ± 0.0322,0.7825 ± 0.0319,0.7868 ± 0.0329,0.7815 ± 0.0322
0,LogReg,0.7795 ± 0.0167,0.7432 ± 0.0213,0.7734 ± 0.0303,0.7795 ± 0.0167
1,SVM-RBF,0.7345 ± 0.0037,0.6264 ± 0.0084,0.7243 ± 0.1321,0.7345 ± 0.0037
2,KNN,0.7175 ± 0.0162,0.6649 ± 0.0228,0.6617 ± 0.0330,0.7175 ± 0.0162
9,MLP,0.6155 ± 0.2391,0.5344 ± 0.2920,0.5465 ± 0.3289,0.6155 ± 0.2391


In [11]:
import pandas as pd

label_col = target_col   # Adult dataset target column: income

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

real_data = processed_data.copy()

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=train_real,
    test_df=test_real,
    label_col=target_col,
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy \u00b1 SD",
            "F1 \u00b1 SD",
            "Precision \u00b1 SD",
            "Recall \u00b1 SD"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"{synth_name} not found in synthetic_datasets. Skipping.")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name].copy()

    # Ensure the target column in the synthetic data is integer-encoded (0 or 1)
    # consistent with the real_data. The `label_encoders` dictionary from previous cells
    # contains the LabelEncoder fitted on the original string labels.
    if label_col in synthetic_train_df.columns:
        if pd.api.types.is_object_dtype(synthetic_train_df[label_col]):
            # If the column contains string labels, transform them to integers
            synthetic_train_df[label_col] = label_encoders[label_col].transform(synthetic_train_df[label_col])
        elif not pd.api.types.is_integer_dtype(synthetic_train_df[label_col]):
            # If it's numeric but not integer (e.g., float), convert to integer
            synthetic_train_df[label_col] = synthetic_train_df[label_col].astype(int)

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=test_real,
        label_col=target_col,
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy \u00b1 SD",
                "F1 \u00b1 SD",
                "Precision \u00b1 SD",
                "Recall \u00b1 SD"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=('_TRTR', '_TSTR')
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy \u00b1 SD_TRTR",
                "Accuracy \u00b1 SD_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
1,SVM-RBF,0.5275 ± 0.0299,0.4220 ± 0.0475,0.4296 ± 0.1323,0.5275 ± 0.0299
7,GradientBoost,0.5250 ± 0.0527,0.4934 ± 0.0538,0.4830 ± 0.0611,0.5250 ± 0.0527
8,AdaBoost,0.5150 ± 0.0489,0.4459 ± 0.0490,0.4205 ± 0.0744,0.5150 ± 0.0489
0,LogReg,0.5050 ± 0.0483,0.3947 ± 0.0506,0.3328 ± 0.0535,0.5050 ± 0.0483
5,RandomForest,0.4875 ± 0.0637,0.4637 ± 0.0680,0.4624 ± 0.0761,0.4875 ± 0.0637
3,NaiveBayes,0.4850 ± 0.0580,0.3885 ± 0.0493,0.3749 ± 0.1033,0.4850 ± 0.0580
6,ExtraTrees,0.4775 ± 0.0650,0.4691 ± 0.0680,0.4693 ± 0.0747,0.4775 ± 0.0650
2,KNN,0.4700 ± 0.0780,0.4465 ± 0.0728,0.4380 ± 0.0805,0.4700 ± 0.0780
9,MLP,0.4275 ± 0.0595,0.4209 ± 0.0583,0.4272 ± 0.0608,0.4275 ± 0.0595
4,DecisionTree,0.4175 ± 0.0602,0.4187 ± 0.0598,0.4406 ± 0.0678,0.4175 ± 0.0602


CTGAN - TSTR


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
1,SVM-RBF,0.4775 ± 0.0142,0.3196 ± 0.0227,0.2897 ± 0.1081,0.4775 ± 0.0142
0,LogReg,0.4750 ± 0.0000,0.3059 ± 0.0000,0.2256 ± 0.0000,0.4750 ± 0.0000
8,AdaBoost,0.4750 ± 0.0000,0.3059 ± 0.0000,0.2256 ± 0.0000,0.4750 ± 0.0000
6,ExtraTrees,0.4725 ± 0.0362,0.3650 ± 0.0448,0.3472 ± 0.0773,0.4725 ± 0.0362
7,GradientBoost,0.4550 ± 0.0387,0.3276 ± 0.0397,0.2916 ± 0.0836,0.4550 ± 0.0387
3,NaiveBayes,0.4475 ± 0.0463,0.3291 ± 0.0471,0.2812 ± 0.0686,0.4475 ± 0.0463
5,RandomForest,0.4425 ± 0.0442,0.3415 ± 0.0468,0.3076 ± 0.0867,0.4425 ± 0.0442
9,MLP,0.4350 ± 0.0709,0.3670 ± 0.0531,0.3388 ± 0.0594,0.4350 ± 0.0709
2,KNN,0.4000 ± 0.0773,0.3365 ± 0.0668,0.3014 ± 0.0721,0.4000 ± 0.0773
4,DecisionTree,0.3300 ± 0.0405,0.3365 ± 0.0395,0.3568 ± 0.0509,0.3300 ± 0.0405


CTGAN - TRTR vs TSTR


...

,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
5,RandomForest,0.4450 ± 0.0511,0.3548 ± 0.0462,0.3017 ± 0.0451,0.4450 ± 0.0511
7,GradientBoost,0.4400 ± 0.0337,0.3299 ± 0.0301,0.2784 ± 0.0491,0.4400 ± 0.0337
8,AdaBoost,0.4025 ± 0.0520,0.3416 ± 0.0429,0.3014 ± 0.0393,0.4025 ± 0.0520
4,DecisionTree,0.3825 ± 0.0590,0.3279 ± 0.0480,0.3022 ± 0.0501,0.3825 ± 0.0590
9,MLP,0.3775 ± 0.0837,0.3277 ± 0.0723,0.2986 ± 0.0656,0.3775 ± 0.0837
1,SVM-RBF,0.3750 ± 0.0553,0.3335 ± 0.0488,0.3124 ± 0.0446,0.3750 ± 0.0553
0,LogReg,0.3725 ± 0.0492,0.3309 ± 0.0434,0.3120 ± 0.0434,0.3725 ± 0.0492
6,ExtraTrees,0.3650 ± 0.0603,0.3050 ± 0.0489,0.2642 ± 0.0434,0.3650 ± 0.0603
3,NaiveBayes,0.3425 ± 0.0800,0.3039 ± 0.0699,0.3078 ± 0.0649,0.3425 ± 0.0800
2,KNN,0.3375 ± 0.0710,0.2918 ± 0.0607,0.2702 ± 0.0553,0.3375 ± 0.0710


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy ± SD_TRTR,Accuracy ± SD_TSTR
0,CTABGAN,SVM-RBF,0.1525,0.088542,0.117206,0.1525,0.5275 ± 0.0299,0.3750 ± 0.0553
1,CTABGAN,GradientBoost,0.0850,0.163441,0.204629,0.0850,0.5250 ± 0.0527,0.4400 ± 0.0337
2,CTABGAN,AdaBoost,0.1125,0.104293,0.119060,0.1125,0.5150 ± 0.0489,0.4025 ± 0.0520
3,CTABGAN,LogReg,0.1325,0.063795,0.020861,0.1325,0.5050 ± 0.0483,0.3725 ± 0.0492
4,CTABGAN,RandomForest,0.0425,0.108945,0.160716,0.0425,0.4875 ± 0.0637,0.4450 ± 0.0511
5,CTABGAN,NaiveBayes,0.1425,0.084543,0.067079,0.1425,0.4850 ± 0.0580,0.3425 ± 0.0800
6,CTABGAN,ExtraTrees,0.1125,0.164138,0.205054,0.1125,0.4775 ± 0.0650,0.3650 ± 0.0603
7,CTABGAN,KNN,0.1325,0.154687,0.167892,0.1325,0.4700 ± 0.0780,0.3375 ± 0.0710
8,CTABGAN,MLP,0.0500,0.093204,0.128583,0.0500,0.4275 ± 0.0595,0.3775 ± 0.0837
9,CTABGAN,DecisionTree,0.0350,0.090887,0.138404,0.0350,0.4175 ± 0.0602,0.3825 ± 0.0590


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
4,TVAE,-0.00225,0.034980,0.045216,-0.00225
2,CopulaGAN,0.03850,0.104647,0.138648,0.03850
1,CTGAN,0.04275,0.102880,0.131278,0.04275
3,GaussianCopula,0.07650,0.126720,0.146804,0.07650
0,CTABGAN,0.09975,0.111648,0.132948,0.09975
5,WGAN_GP,0.15650,0.149910,0.101390,0.15650


In [12]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
